In [ ]:
import typing
import random
import itertools
import json

import tqdm
import pandas as pd
import seaborn as sns

import _lib

In [ ]:
MODEL: str = "llama2:70b-chat-q6_K"  # "mistral:7b-instruct-v0.2-q6_K",   "llama3:70b-instruct-q6_K"
MAX_HISTORY_LENGTH: int = 10

In [ ]:
inference = _lib.Inference(model=MODEL)

data: typing.Dict = json.load(open("setup/experiment.json"))
prompt: str = open("setup/prompt.txt").read()

In [ ]:
configurations: typing.List[typing.Tuple] = list(
    itertools.product(*[
        data["personas"].items(), 
        data["questionnaire"]["statements"].items(),
        {
            (n, m := MAX_HISTORY_LENGTH - n): random.sample([
                *random.sample(data["statements"]["pro"], n),
                *random.sample(data["statements"]["contra"], m)
            ], n+m)
            for n in range (0, MAX_HISTORY_LENGTH + 1)
        }.items()
    ])
)
len(configurations)

In [ ]:
results: typing.List = []

for (persona_label, persona), (statement_id, statement), (history_distribution, history) in tqdm.tqdm(configurations):
        results.append({
            "statement": statement_id,
            "persona": persona_label,
            "history": {
                  "num_pro": history_distribution[0],
                  "num_contra":  history_distribution[1],  
            },
            "response": inference(
                prompt=prompt.format(
                    persona=persona, 
                    history="\n".join(history), 
                    task=data["questionnaire"]["task"], 
                    statement=statement, 
                    scale=data["questionnaire"]["scale"]
                )
            ).extract_numeric_answer()
        })
        

In [ ]:
results_df = (
    pd.json_normalize(results)
    .pipe(lambda _df: _df.assign(repsonse=pd.Categorical(
        _df["response"], categories=["1", "2", "3", "4", "5"], ordered=True
    )))
)
results_df.to_json(f"results/{MODEL.replace(":", "-")}.json")

In [ ]:
g = sns.PairGrid(
    data=results_df.pivot(index=["persona", "history.num_pro"], columns=["statement"], values="response").reset_index(),
    hue="persona",
    x_vars=["1", "2", "3", "4", "5"], 
    y_vars=["history.num_pro"],
    height=8, 
    aspect=.25
)

g.map(
    sns.stripplot, 
    size=10, 
    orient="h", 
    dodge=True,
    palette="flare_r", 
    linewidth=1, 
    edgecolor="w",
)
for ax in g.axes.flat:

    ax.xaxis.grid(True)
    ax.yaxis.grid(True)


sns.despine(left=True, bottom=True)
g.add_legend()
g.figure.savefig(f"results/{MODEL.replace(":", "-")}.pdf")